# **Contextual RAG**

Contextual Retrieval-Augmented Generation (RAG) is an advanced RAG technique that improves response relevance and efficiency by incorporating contextual compression during the retrieval process. Traditional RAG retrieves and sends full documents to the generation model, which may include irrelevant information, leading to higher costs and less accurate responses.

In Contextual RAG, the retrieved documents are processed through a Document Compressor before being passed to the language model. This compressor extracts and retains only the most relevant information for the query, or even discards entire irrelevant documents. This approach reduces the noise in the retrieved context, resulting in more precise, concise, and cost-effective responses from the generation model.

Contextual RAG(문맥적 검색 증강 생성)는 검색 과정에서 문맥적 압축(contextual compression)을 통합하여 응답의 관련성과 효율성을 향상시키는 고급 RAG 기법입니다. 기존 RAG는 전체 문서를 검색하여 생성 모델에 전달하는데, 이 과정에서 관련 없는 정보가 포함될 수 있어 비용 증가와 응답 정확도 저하로 이어질 수 있습니다.

Contextual RAG에서는 검색된 문서가 언어 모델에 전달되기 전에 문서 압축기(Document Compressor)를 거치게 됩니다. 이 압축기는 쿼리에 가장 관련성 높은 정보만 추출하여 보존하거나, 완전히 관련 없는 문서는 아예 제거합니다. 이 방식은 검색된 문맥에서 노이즈를 줄여, 생성 모델로부터 더 정확하고 간결하며 비용 효율적인 응답을 이끌어냅니다.

Reference: [Contextual RAG](https://python.langchain.com/docs/how_to/contextual_compression/)

## **Indexing**

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
# create vectorstore
from langchain_chroma import Chroma
import chromadb 

chroma_client = chromadb.PersistentClient(path="../database/chroma_db")
vectorstore = Chroma(
    client=chroma_client,
    collection_name="langchain",  # 기존에 저장한 컬렉션 이름
    embedding_function=embeddings,
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## **Retriever**

In [4]:
# create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",        # 검색 방식
    search_kwargs={"k": 4}           # 검색 세부 옵션 (딕셔너리)
)

## **Contextual Retriever**

In [5]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
    )

In [6]:
# create compression retriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [8]:
# checking compressed doc
compressed_docs = compression_retriever.invoke("what are points on a mortgage")
compressed_docs

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(metadata={'row': 1, 'source': '../data/context.csv'}, page_content='Discount points, also called mortgage points or simply points, are a form of pre-paid interest available in the United States when arranging a mortgage. One point equals one percent of the loan amount. By charging a borrower points, a lender effectively increases the yield on the loan above the amount of the stated interest rate.  Borrowers can offer to pay a lender points as a method to reduce the interest rate on the loan, thus obtaining a lower monthly payment in exchange for this'),
 Document(metadata={'row': 1, 'source': '../data/context.csv'}, page_content="points is the concept of the 'no closing cost loan', in which the consumer accepts a higher interest rate in return for the lender paying the loan's closing costs up front.  In some cases a purchaser can negotiate with the seller to get them to pay seller's points which can be used to pay mortgage points."),
 Document(metadata={'row': 1, 'source': '.

## **RAG Chain**

In [9]:
# create document chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.
Context: {context}

Question: {input}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": compression_retriever, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
# response
response = rag_chain.invoke("what are points on a mortgage")
response

'Points on a mortgage, also called discount points or mortgage points, are a form of pre-paid interest in the United States. One point equals one percent of the loan amount. Borrowers can pay points to the lender to reduce the interest rate on the loan, which lowers the monthly payment. Each point purchased typically reduces the loan interest rate by about 0.125% to 0.25%. Points can also be used to help qualify for a loan by reducing the monthly payment.'

## **Preparing Data for Evaluation**

In [11]:
# create dataset
questions = ["what are points on a mortgage"]

data = {"query": [], "response": [], "context": []}

for query in questions:
    resp = rag_chain.invoke(query)
    context = [doc.page_content for doc in compression_retriever.invoke(query)]
    data["query"].append(query)
    data["response"].append(resp)
    data["context"].append(context)

import pandas as pd
df = pd.DataFrame(data)
df

,query,response,context
0,what are points on a mortgage,"Points on a mortgage, also called discount poi...","[Discount points, also called mortgage points ..."


In [12]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy

eval_data = [
    {
        "user_input": q,
        "response": r,
        "retrieved_contexts": c,
    }
    for q, r, c in zip(data["query"], data["response"], data["context"])
]
ragas_dataset = EvaluationDataset.from_dict(eval_data)

In [13]:
# create dataframe
result = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(), AnswerRelevancy()],
)
print(result)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'answer_relevancy': 0.9304}


In [14]:
result.to_pandas()

,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,what are points on a mortgage,"[Discount points, also called mortgage points ...","Points on a mortgage, also called discount poi...",1.0,0.930383
